In [2]:
import time

%matplotlib qt5
import numpy as np
import matplotlib.pyplot as mplt

from siriuspy.devices import FamBPMs
from apsuite.commisslib.impedance_ivu_meas import ImpedanceIVUMeas

# Define functions

# Create Devices

In [ ]:
meas = ImpedanceIVUMeas()
fam_bpms = meas.devices['fambpms']
evg = meas.devices['evg']
sofb = meas.devices['sofb']

In [ ]:
meas.connected

(True, True)

# Procedures Pre-measurement

## Set attenuation of all BPMs

 Amplitude of signal in BPMs antennas should be ~10% of full-scale, which is ~32k. So, an amplitude of ~3k is a good choice. Measurements with some BPMs showed that an attenuation of 6 is enough to get this amplitude with ~1mA in the high charge bunch

In [5]:
fam_bpms_ids = FamBPMs(FamBPMs.DEVICES.SI_ID)

In [6]:
att = 16

In [ ]:
fam_bpms.set_attenuation(att)
fam_bpms_ids.set_attenuation(att)

True

In [ ]:
for bpm in fam_bpms.bpms + fam_bpms_ids.bpms:
    if bpm.rffe_att != att:
        print(bpm.devname)

## Turn  off tune measurement systems

 - all drivers from 3 bbb
 - horizontal and vertical spectrum analyzers

## Turn off switching from BPMs

In [ ]:
fam_bpms.set_switching_mode(mode='direct')

## Turn off Post-Mortem acquisitions to reduce memory use in AFCs

In [6]:
fam_bpms_psmtm = FamBPMs(FamBPMs.DEVICES.SI, ispost_mortem=True)

/usr/local/lib/python3.6/site-packages/epics/ca.py:1507: UserWarning: ca.get('SI-09C1:DI-BPM-1:PMPosXData') timed out after 1.00 seconds.
  warnings.warn(msg % (name(chid), timeout))


In [8]:
fam_bpms_psmtm.cmd_stop_mturn_acquisition()

0

## Test data intregrity

In [ ]:
onof = False
for bpm in fam_bpms.bpms:
    bpm['ADCTestDataEn-Sel'] = onof

In [ ]:
for bpm in fam_bpms.bpms:
    if bpm['ADCTestDataEn-Sts'] != onof:
        print(bpm.devname)

In [ ]:
meas.params.nrturns = 500
meas.params.num_acquisitions = 1
meas.params.acq_rate = 'ADC'
meas.params.timeout = 40
print(meas.params)

AcqBPMsSignalsParams:
    trigbpm_delay              = same       (current value will not be changed)
    trigbpm_nrpulses           =         1  
    do_pulse_evg               = True       
    timing_event               = Study      
    event_delay                = same       (current value will not be changed)
    event_mode                 = Injection  
    timeout                    = 40.000000  [s]
    nrpoints_before            =         0  
    nrpoints_after             =    191000  
    acq_rate                   = ADC        
    acq_repeat                 =         0  
    signals2acq                = ABCD       

MeasIVUImpedanceParams:
    num_acquisitions = 1
    save_raw_data = False
    num_buckets_to_process = 2
    nrturns = 500



In [ ]:
meas.start()

Acquisition 01/01
it took 10.565164s to update bpms
Finished!


In [ ]:
meas.save_data('rate_adc_test_data_n0')

In [ ]:
data = np.array([meas.data[0]['ampl' + a] for a in 'abcd'])

In [16]:
diff = np.diff(data, axis=1)
np.array_equal(np.sum(diff != 1, axis=1), np.sum(diff < -10000, axis=1))

True

## Make some measurements to check precision

In [11]:
meas.params.num_acquisitions = 1
meas.params.nrturns = 500
meas.params.timeout = 40

print(meas.params)

AcqBPMsSignalsParams:
    trigbpm_delay              = same       (current value will not be changed)
    trigbpm_nrpulses           =         1  
    do_pulse_evg               = True       
    timing_event               = Study      
    event_delay                = same       (current value will not be changed)
    event_mode                 = Injection  
    timeout                    = 40.000000  [s]
    nrpoints_before            =         0  
    nrpoints_after             =    191000  
    acq_rate                   = ADCSwp     
    acq_repeat                 =         0  
    signals2acq                = ABCD       

MeasIVUImpedanceParams:
    num_acquisitions = 1
    save_raw_data = False
    num_buckets_to_process = 2
    nrturns = 500



In [12]:
meas.prepare_timing()

In [13]:
meas.start()

Acquisition 01/01
it took 8.164553s to update bpms
Finished!


In [ ]:
meas.stop()

In [9]:
meas.ismeasuring

False

In [ ]:
meas.save_data('test_single_acq_n0')

## Test Different data processing algorithms

In [3]:
anl = ImpedanceIVUMeas(isonline=False)

In [10]:
folder = '/home/fernando/shared/screens-iocs/data_by_day/2026-08-17-SI_IVU_impedance_measurement/'
anl.load_and_apply(folder + 'data_initial_tests/test_single_acq_n1.pickle')
anl.process_data(return_all=True)

In [5]:
sofb_ref = anl.calc_sofb_orbit(isref=True)
sofb_orb = anl.calc_sofb_orbit(isref=False)

In [6]:
anl.process_data(return_all=True, proctype=1)
dorb1, orb1_b1, orb1_b2 = anl.calc_delta_orbit_2_bunches()
curr1_b1, curr1_b2 = anl.calc_current_2_bunches()

In [7]:
anl.process_data(return_all=True, proctype=2)
dorb2, orb2_b1, orb2_b2 = anl.calc_delta_orbit_2_bunches()
curr2_b1, curr2_b2 = anl.calc_current_2_bunches()

In [8]:
anl.process_data(return_all=True, proctype=3)
dorb3, orb3_b1, orb3_b2 = anl.calc_delta_orbit_2_bunches()
curr3_b1, curr3_b2 = anl.calc_current_2_bunches()

In [9]:
dorb1.shape, dorb2.shape, dorb3.shape

((1, 160, 500), (1, 160, 500), (1, 160, 500))

In [10]:
dt1 = orb1_b2
dt2 = orb2_b2
dt3 = orb3_b2
axes = (0, -1)
fig, (ax, ay) = mplt.subplots(2, 1, sharex=True)
ay.plot(dt1.mean(axis=axes) - dt2.mean(axis=axes), label='1-2')
ay.plot(dt1.mean(axis=axes) - dt3.mean(axis=axes), label='1-3')
ay.plot(dt2.mean(axis=axes) - dt3.mean(axis=axes), label='2-3')
ax.plot(dt1.std(axis=axes), label='Proc1')
ax.plot(dt2.std(axis=axes), label='Proc2')
ax.plot(dt3.std(axis=axes), label='Proc3')
ax.legend(loc='best')
ay.legend(loc='best')
fig.show()

In [11]:
curr1_b1.shape, curr2_b1.shape, curr3_b1.shape

((1, 80, 500), (1, 80, 500), (1, 80, 500))

In [15]:
fig, (ax, ay) = mplt.subplots(2, 1, sharex=True)
acqs = np.arange(curr1_b1.shape[1])

def fun1(curr):
    return curr.mean(axis=(0, 2))

def fun2(curr):
    return curr.std(axis=(0, 2))


lin = ax.plot(acqs, fun1(curr1_b1), 'o', color='C0')[0]
lin.set_label('Proc1')
lin = ax.plot(acqs, fun1(curr2_b1), 'o', color='C1')[0]
lin.set_label('Proc2')
lin = ax.plot(acqs, fun1(curr3_b1), 'o', color='C2')[0]
lin.set_label('Proc3')
ax.legend(loc='best')

lin = ay.plot(acqs, fun2(curr1_b1), 'o', color='C0')[0]
lin.set_label('Proc1')
lin = ay.plot(acqs, fun2(curr2_b1), 'o', color='C1')[0]
lin.set_label('Proc2')
lin = ay.plot(acqs, fun2(curr3_b1), 'o', color='C2')[0]
lin.set_label('Proc3')
ay.legend(loc='best')

fig.show()

In [31]:
ss = 8
nch = 12
ncv = 16
idcs = sofb.find_closest_corrs_ss(ss, nch, ncv)

In [53]:
# dorb -= 1*dorb.mean(axis=-1)[:, None]
# dorb = sofb.filter_local_distortions(idcs, dorb=dorb)
dorb = orb1_b2
mean = dorb.mean(axis=-1)
std = dorb.std(axis=-1) / np.sqrt(dorb.shape[-1])
fig, ax = mplt.subplots()
ax.plot(dorb, color='C0', alpha=0.3)
ax.plot(mean, color='k', lw=2)
ax.plot(mean + std, '--', color='k', lw=2)
ax.plot(mean - std, '--', color='k', lw=2)
fig.show()

In [9]:
anl.data[0]['ant_raw'].shape

(4, 80, 191000)

In [16]:
fig, ax = mplt.subplots()
bpms = [0, 16, 32, 120]
lins = ax.plot(anl.data[0]['ant_raw'][0, bpms, :1910].T + np.arange(4)[None, :] * 2000)
[l.set_label(f'BPM {bpms[i]}') for i, l in enumerate(lins)]
lins = [ax.axvline(382 * i, ls='--', color='k') for i in range(6)]
lins[0].set_label('Turns')
ax.set_ylabel('Raw Signal Antenna A [cnts]')
ax.set_xlabel('Sample Index')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1), ncols=5)
fig.tight_layout()
fig.show()

In [24]:
fig, ax = mplt.subplots()
bpms = [0, 16, 32, 120]

ax.axvspan(xmin=50-16, xmax=50+40, color='blue', alpha=0.1)
ax.axvspan(xmin=284-16, xmax=284+40, color='red', alpha=0.1)
ax.annotate('Bunch 1', (36, 3000), color='blue', weight='bold', textcoords='data')
ax.annotate('Bunch 2', (270, 3000), color='red', weight='bold', textcoords='data')

lins = ax.plot(anl.data[0]['ant_raw2'][0, bpms, 0].T + np.arange(4)[None, :] * 2000)
[l.set_label(f'BPM {bpms[i]}') for i, l in enumerate(lins)]
ax.set_ylabel('Raw Signal Antenna A [cnts]')
ax.set_xlabel('Sample Index')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1), ncols=5)
fig.tight_layout()
fig.show()

In [75]:
import scipy.signal as scy_sig

In [124]:
filt = scy_sig.butter(6, 0.1, fs=1, btype='low', output='sos')
amp_filt = scy_sig.sosfiltfilt(filt, anl.data[0]['ant_amp2'][:, 0].reshape(-1, 382 * 5), axis=-1)

In [125]:
fig, ax = mplt.subplots()
ax.plot(anl.data[0]['ant_amp2'][:, 0].reshape(-1, 382 * 5)[0])
ax.plot(amp_filt[0])
[ax.axvline(382 * i, ls='--', color='k') for i in range(6)]
fig.show()

In [32]:
fig, ax = mplt.subplots()
ax.plot(anl.data[0]['ant_amp2_mean'].reshape(-1, 382).T)
fig.show()

In [26]:
fig, ax = mplt.subplots()
ax.plot(anl.data[0]['ant_amp2_std'].reshape(-1, 382).T)
fig.show()

# Make measurement

In [ ]:
meas.params.num_acquisitions = 5
meas.params.nrturns = 500
meas.params.timeout = 40
print(meas.params)

Odd BPMs
AcqBPMsSignalsParams:
    trigbpm_delay              = same       (current value will not be changed)
    trigbpm_nrpulses           =         1  
    do_pulse_evg               = True       
    timing_event               = Study      
    event_delay                = same       (current value will not be changed)
    event_mode                 = Injection  
    timeout                    = 40.000000  [s]
    nrpoints_before            =         0  
    nrpoints_after             =    191000  
    acq_rate                   = ADCSwp     
    acq_repeat                 =         0  
    signals2acq                = ABCD       

MeasIVUImpedanceParams:
    num_acquisitions = 5
    save_raw_data = False
    num_buckets_to_process = 2
    nrturns = 500
    bucket_hi_charge = 1
    bucket_lo_charge = 530

Even BPMs
AcqBPMsSignalsParams:
    trigbpm_delay              = same       (current value will not be changed)
    trigbpm_nrpulses           =         1  
    do_pulse_evg     

In [ ]:
meas.prepare_timing()

In [ ]:
meas.start()

Acquisition 01/05
it took 8.049106s to update bpms
Acquisition 02/05
it took 8.126437s to update bpms
Acquisition 03/05
it took 8.106949s to update bpms
Acquisition 04/05
it took 8.048394s to update bpms
Acquisition 05/05
it took 7.972570s to update bpms
Finished!


In [ ]:
meas.stop()

In [ ]:
meas.ismeasuring

False

In [ ]:
meas.save_data(
    'ivu_08_gap_8p600mm_bump_y_p1000um_yp_p0000urad_acq_n0'
)

# Analyse Measurements

In [2]:
folder = '/home/fernando/shared/screens-iocs/data_by_day/'
folder += '2026-08-17-SI_IVU_impedance_measurement/data_ivu_08_gap_4p300mm/'

In [ ]:
fil_fmt = 'ivu_08_gap_4p300mm_bump_y_{}um_yp_p0000urad_acq_n0.pickle'

In [4]:
anl = ImpedanceIVUMeas(isonline=False)

In [ ]:
bumps = np.array([-1000, -500, 0, 500, 1000])
dorbs, orbs_b1, orbs_b2 = [], [], []
sumst = []
sums1 = []
sums2 = []
for bump in bumps:
    bmp_str = '{:04d}'.format(abs(bump))
    bmp_str = ('p' if bump >= 0 else 'm') + bmp_str
    dorb = np.zeros((5, 320, 500))
    orb_b1 = np.zeros((5, 320, 500))
    orb_b2 = np.zeros((5, 320, 500))
    sumt = np.zeros((5, 160))
    sum1 = np.zeros((5, 160))
    sum2 = np.zeros((5, 160))

    fil = fil_fmt.format(bmp_str)
    anl.load_and_apply(folder + fil)
    dorb_, orb_b1_, orb_b2_ = anl.calc_delta_orbit_2_bunches()
    curr_b1, curr_b2 = anl.calc_current_2_bunches()
    sum_t, sum_b1, sum_b2 = anl.calc_sum_signal_2_bunches()
    dorb[:, i::2] = dorb_
    orb_b1[:, i::2] = orb_b1_
    orb_b2[:, i::2] = orb_b2_
    sumt[:, i::2] = sum_t.sum(axis=1)
    sum1[:, i::2] = sum_b1.sum(axis=1)
    sum2[:, i::2] = sum_b2.sum(axis=1)
    dorbs.append(dorb)
    orbs_b1.append(orb_b1)
    orbs_b2.append(orb_b2)
    sumst.append(sumt)
    sums1.append(sum1)
    sums2.append(sum2)

filt_fun = ImpedanceIVUMeas.filter_switching_cycles
# filt_fun = lambda x, y, z, axis: x
dorbs = filt_fun(np.array(dorbs), 48, 1, axis=-1)
orbs_b1 = filt_fun(np.array(orbs_b1), 48, 1, axis=-1)
orbs_b2 = filt_fun(np.array(orbs_b2), 48, 1, axis=-1)
sumst = np.array(sumst)
sums1 = np.array(sums1)
sums2 = np.array(sums2)

In [ ]:
def fun(dorbs):
    # dmean = dorbs.mean(axis=(1, 3))
    # dmean -= dmean[2]
    # return dmean.T

    # return np.diff(dorbs.mean(axis=-1)[4], axis=0).T
    return dorbs[0].mean(axis=(0, 2))
    return dorbs[0].std(axis=(0, 2))
    # return np.abs(np.fft.rfft(dorbs[0, 0, 0]))
    # return np.diff(dorbs[0, 0, 0], axis=-1)

fig, ax = mplt.subplots()
ax.plot(fun(dorbs), "C0")
ax.plot(fun(orbs_b1), 'C1')
ax.plot(fun(orbs_b2), "C2")
fig.show()

In [15]:
fig, ax = mplt.subplots()
ax.plot(sumst.reshape(-1, 160).T, color='C0')
ax.plot(sums1.reshape(-1, 160).T, color='C1')
ax.plot(sums2.reshape(-1, 160).T, color='C2')
fig.show()

In [12]:
sumst.shape

(5, 5, 160)